# 01 — Audit des sources et consolidation du dataset annexe

Ce notebook a pour objectif de :

- recenser les sources CSV mobilisées dans le projet ;
- contrôler leur structure et leur compatibilité ;
- harmoniser les colonnes nécessaires à la consolidation ;
- produire le dataset annexe utilisé ensuite dans le notebook de modélisation du **modèle 2**.

## Positionnement dans la démarche

Le dataset principal `crop_yield.csv` est exploré dans le notebook `02_eda_analysis.ipynb`.  
Le présent notebook se concentre sur les **sources tabulaires du dataset annexe** issues du dossier *Crop Yield Prediction Dataset*.

## Entrées

- fichiers CSV du dossier `data/raw_local/...`

## Sortie

- `data/processed/yield_dataset_consolidated.csv`

## Décision attendue

La sortie attendue est un fichier consolidé, propre et documenté, prêt à être utilisé dans le notebook `04_modeling_secondary_recommendation.ipynb`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from project_paths import RAW_DIR, PROCESSED_DIR
from src.data_manage.load_data import load_csv, list_raw_files

import numpy as np
import pandas as pd
from IPython.display import display


## 1. Inventaire des sources disponibles

Cette première étape recense les fichiers CSV présents dans le répertoire des données brutes.
Elle permet de vérifier rapidement l’organisation du projet avant de cibler les sources effectivement mobilisées dans la consolidation.


In [2]:
csv_files = sorted(list_raw_files("*.csv"))
inventory = pd.DataFrame({
    "relative_path": [str(file.relative_to(RAW_DIR)) for file in csv_files],
    "filename": [file.name for file in csv_files],
})

print(f"Nombre total de fichiers CSV détectés : {len(inventory)}")
display(inventory)


Nombre total de fichiers CSV détectés : 6


,relative_path,filename
0,Agriculture Crop Yield\crop_yield.csv,crop_yield.csv
1,Crop Yield Prediction Dataset\pesticides.csv,pesticides.csv
2,Crop Yield Prediction Dataset\rainfall.csv,rainfall.csv
3,Crop Yield Prediction Dataset\temp.csv,temp.csv
4,Crop Yield Prediction Dataset\yield.csv,yield.csv
5,Crop Yield Prediction Dataset\yield_df.csv,yield_df.csv


## 2. Chargement ciblé des sources utiles au dataset annexe

Le dataset annexe est reconstruit à partir de quatre sources élémentaires :

- `yield.csv` pour le rendement ;
- `pesticides.csv` pour l’usage de pesticides ;
- `rainfall.csv` pour la pluie ;
- `temp.csv` pour la température.

Le fichier `yield_df.csv` est également chargé comme **référence de contrôle** afin de comparer le résultat reconstruit avec une version déjà fusionnée.


In [3]:
yield_raw = load_csv("Crop Yield Prediction Dataset/yield.csv")
pest_raw = load_csv("Crop Yield Prediction Dataset/pesticides.csv")
rain_raw = load_csv("Crop Yield Prediction Dataset/rainfall.csv")
temp_raw = load_csv("Crop Yield Prediction Dataset/temp.csv")
yield_ref = load_csv("Crop Yield Prediction Dataset/yield_df.csv")

sources = {
    "yield_raw": yield_raw,
    "pesticides_raw": pest_raw,
    "rainfall_raw": rain_raw,
    "temp_raw": temp_raw,
    "yield_ref": yield_ref,
}


## 3. Inspection synthétique des sources

L’objectif n’est pas de produire une EDA complète à ce stade, mais de vérifier les propriétés structurelles indispensables à la fusion :

- dimensions ;
- noms de colonnes ;
- types ;
- valeurs manquantes ;
- doublons sur les clés métier attendues.


In [4]:
def source_overview(name: str, df: pd.DataFrame, key_cols: list[str] | None = None) -> pd.Series:
    overview = {
        "source": name,
        "n_rows": df.shape[0],
        "n_cols": df.shape[1],
        "n_missing_total": int(df.isna().sum().sum()),
        "columns": ", ".join(df.columns.astype(str).tolist()),
    }
    if key_cols is not None and set(key_cols).issubset(df.columns):
        overview["duplicate_keys"] = int(df.duplicated(subset=key_cols).sum())
        overview["key_cols"] = ", ".join(key_cols)
    else:
        overview["duplicate_keys"] = np.nan
        overview["key_cols"] = ""
    return pd.Series(overview)

overview_df = pd.DataFrame([
    source_overview("yield_raw", yield_raw, ["Area", "Item", "Year"]),
    source_overview("pesticides_raw", pest_raw, ["Area", "Year"]),
    source_overview("rainfall_raw", rain_raw, [" Area", "Year"]),
    source_overview("temp_raw", temp_raw, ["country", "year"]),
    source_overview("yield_ref", yield_ref, ["Area", "Item", "Year"]),
])

display(overview_df)


,source,n_rows,n_cols,n_missing_total,columns,duplicate_keys,key_cols
0,yield_raw,56717,12,0,"Domain Code, Domain, Area Code, Area, Element ...",0,"Area, Item, Year"
1,pesticides_raw,4349,7,0,"Domain, Area, Element, Item, Year, Unit, Value",0,"Area, Year"
2,rainfall_raw,6727,3,774,"Area, Year, average_rain_fall_mm_per_year",0,"Area, Year"
3,temp_raw,71311,3,2547,"year, country, avg_temp",42797,"country, year"
4,yield_ref,28242,8,0,"Unnamed: 0, Area, Item, Year, hg/ha_yield, ave...",15112,"Area, Item, Year"


In [5]:
for name, df in sources.items():
    print("=" * 100)
    print(name)
    display(df.head())
    print("Types :")
    display(df.dtypes.rename("dtype").to_frame())
    print("Valeurs manquantes :")
    display(df.isna().sum().rename("n_missing").to_frame().sort_values("n_missing", ascending=False).head(10))


yield_raw


,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value
0,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1961,1961,hg/ha,14000
1,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1962,1962,hg/ha,14000
2,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1963,1963,hg/ha,14260
3,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1964,1964,hg/ha,14257
4,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1965,1965,hg/ha,14400


Types :


,dtype
Domain Code,object
Domain,object
Area Code,int64
Area,object
Element Code,int64
Element,object
Item Code,int64
Item,object
Year Code,int64
Year,int64


Valeurs manquantes :


...

,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


Types :


,dtype
Unnamed: 0,int64
Area,object
Item,object
Year,int64
hg/ha_yield,int64
average_rain_fall_mm_per_year,float64
pesticides_tonnes,float64
avg_temp,float64


Valeurs manquantes :


,n_missing
Unnamed: 0,0
Area,0
Item,0
Year,0
hg/ha_yield,0
average_rain_fall_mm_per_year,0
pesticides_tonnes,0
avg_temp,0


## 4. Harmonisation des colonnes et sélection des variables utiles

Les fichiers ne partagent pas directement les mêmes conventions de nommage.  
Cette étape aligne les colonnes nécessaires à la fusion et élimine les variables non indispensables à la première version du dataset annexe.


In [6]:
yield_clean = (
    yield_raw
    .rename(columns={"Value": "hg/ha_yield"})
    [["Area", "Item", "Year", "Unit", "hg/ha_yield"]]
    .copy()
)

pest_clean = (
    pest_raw
    .rename(columns={"Value": "pesticides_tonnes"})
    [["Area", "Year", "Unit", "pesticides_tonnes"]]
    .copy()
)

rain_clean = (
    rain_raw
    .rename(columns={" Area": "Area"})
    [["Area", "Year", "average_rain_fall_mm_per_year"]]
    .copy()
)

temp_clean = (
    temp_raw
    .rename(columns={"country": "Area", "year": "Year"})
    [["Area", "Year", "avg_temp"]]
    .copy()
)

print("Colonnes après harmonisation :")
for name, df in {
    "yield_clean": yield_clean,
    "pest_clean": pest_clean,
    "rain_clean": rain_clean,
    "temp_clean": temp_clean,
}.items():
    print(f"- {name}: {df.columns.tolist()}")


Colonnes après harmonisation :
- yield_clean: ['Area', 'Item', 'Year', 'Unit', 'hg/ha_yield']
- pest_clean: ['Area', 'Year', 'Unit', 'pesticides_tonnes']
- rain_clean: ['Area', 'Year', 'average_rain_fall_mm_per_year']
- temp_clean: ['Area', 'Year', 'avg_temp']


## 5. Contrôle des clés de jointure

La fusion repose principalement sur les clés suivantes :

- `Area`
- `Item`
- `Year`

Certaines tables ne sont pas définies au niveau de `Item`.  
Il est donc nécessaire de vérifier la compatibilité effective des domaines sur `Area` et `Year`, puis de traiter le cas particulier des doublons dans la table de température.


In [7]:
key_diagnostics = []

for name, df, cols in [
    ("yield_clean", yield_clean, ["Area", "Item", "Year"]),
    ("pest_clean", pest_clean, ["Area", "Year"]),
    ("rain_clean", rain_clean, ["Area", "Year"]),
    ("temp_clean", temp_clean, ["Area", "Year"]),
]:
    row = {"source": name}
    for col in cols:
        row[f"{col}_nunique"] = df[col].nunique(dropna=True)
    row["duplicate_on_expected_key"] = int(df.duplicated(subset=cols).sum())
    key_diagnostics.append(row)

display(pd.DataFrame(key_diagnostics))


,source,Area_nunique,Item_nunique,Year_nunique,duplicate_on_expected_key
0,yield_clean,212,10.0,56,0
1,pest_clean,168,NaN,27,0
2,rain_clean,217,NaN,31,0
3,temp_clean,137,NaN,271,42797


In [8]:
areas_yield = set(yield_clean["Area"].astype(str).str.strip())
areas_pest = set(pest_clean["Area"].astype(str).str.strip())
areas_rain = set(rain_clean["Area"].astype(str).str.strip())
areas_temp = set(temp_clean["Area"].astype(str).str.strip())

years_yield = set(yield_clean["Year"])
years_pest = set(pest_clean["Year"])
years_rain = set(rain_clean["Year"])
years_temp = set(temp_clean["Year"])

coverage_summary = pd.DataFrame([
    {"comparison": "Yield ∩ Pesticides (Area)", "n_common": len(areas_yield & areas_pest)},
    {"comparison": "Yield ∩ Rainfall (Area)", "n_common": len(areas_yield & areas_rain)},
    {"comparison": "Yield ∩ Temp (Area)", "n_common": len(areas_yield & areas_temp)},
    {"comparison": "Yield ∩ Pesticides (Year)", "n_common": len(years_yield & years_pest)},
    {"comparison": "Yield ∩ Rainfall (Year)", "n_common": len(years_yield & years_rain)},
    {"comparison": "Yield ∩ Temp (Year)", "n_common": len(years_yield & years_temp)},
])

display(coverage_summary)


,comparison,n_common
0,Yield ∩ Pesticides (Area),167
1,Yield ∩ Rainfall (Area),169
2,Yield ∩ Temp (Area),118
3,Yield ∩ Pesticides (Year),27
4,Yield ∩ Rainfall (Year),30
5,Yield ∩ Temp (Year),53


## 6. Traitement du cas particulier de la température

La table de température présente plusieurs lignes pour une même paire `Area`–`Year`.  
Afin de rétablir une granularité compatible avec la fusion, la température moyenne est agrégée au niveau `Area`–`Year`.


In [9]:
temp_dupes = (
    temp_clean
    .groupby(["Area", "Year"], as_index=False)
    .agg(
        n_rows=("avg_temp", "size"),
        n_unique_temp=("avg_temp", "nunique"),
        temp_min=("avg_temp", "min"),
        temp_max=("avg_temp", "max"),
        temp_mean=("avg_temp", "mean"),
    )
)

print("Nombre de clés Area-Year apparaissant plus d'une fois :", int((temp_dupes["n_rows"] > 1).sum()))
display(temp_dupes.loc[temp_dupes["n_rows"] > 1].head(20))


Nombre de clés Area-Year apparaissant plus d'une fois : 7994


,Area,Year,n_rows,n_unique_temp,temp_min,temp_max,temp_mean
871,Argentina,1855,2,2,14.00,14.22,14.110
872,Argentina,1856,2,2,16.23,16.80,16.515
873,Argentina,1857,2,2,16.54,17.33,16.935
874,Argentina,1858,2,2,16.22,16.64,16.430
875,Argentina,1859,2,2,16.79,16.85,16.820
876,Argentina,1860,2,2,16.45,16.46,16.455
877,Argentina,1861,2,2,16.27,16.63,16.450
878,Argentina,1862,2,2,16.32,16.65,16.485
879,Argentina,1863,2,2,15.86,16.16,16.010
880,Argentina,1864,2,2,16.35,16.79,16.570


In [10]:
temp_agg = (
    temp_clean
    .groupby(["Area", "Year"], as_index=False)
    .agg(avg_temp=("avg_temp", "mean"))
)

print("Doublons Area-Year après agrégation :", temp_agg.duplicated(subset=["Area", "Year"]).sum())
display(temp_agg.head())


Doublons Area-Year après agrégation : 0


,Area,Year,avg_temp
0,Afghanistan,1833,13.91
1,Afghanistan,1834,13.91
2,Afghanistan,1835,14.71
3,Afghanistan,1836,NaN
4,Afghanistan,1837,15.47


## 7. Construction du dataset consolidé

La consolidation est réalisée par jointure gauche à partir de `yield_clean`, qui joue le rôle de table pivot.
Le choix de `yield_clean` comme base garantit que la variable cible du modèle secondaire reste au centre de la fusion.


In [11]:
merged = (
    yield_clean
    .merge(
        pest_clean[["Area", "Year", "pesticides_tonnes"]],
        on=["Area", "Year"],
        how="left",
    )
    .merge(
        rain_clean[["Area", "Year", "average_rain_fall_mm_per_year"]],
        on=["Area", "Year"],
        how="left",
    )
    .merge(
        temp_agg[["Area", "Year", "avg_temp"]],
        on=["Area", "Year"],
        how="left",
    )
)

print("Shape après fusion :", merged.shape)
display(merged.head())


Shape après fusion : (56717, 8)


,Area,Item,Year,Unit,hg/ha_yield,pesticides_tonnes,average_rain_fall_mm_per_year,avg_temp
0,Afghanistan,Maize,1961,hg/ha,14000,NaN,NaN,14.23
1,Afghanistan,Maize,1962,hg/ha,14000,NaN,NaN,14.10
2,Afghanistan,Maize,1963,hg/ha,14260,NaN,NaN,15.01
3,Afghanistan,Maize,1964,hg/ha,14257,NaN,NaN,13.73
4,Afghanistan,Maize,1965,hg/ha,14400,NaN,NaN,13.90


## 8. Contrôles qualité après fusion

Les contrôles suivants sont appliqués :

- comptage des valeurs manquantes après jointure ;
- suppression des lignes inexploitables pour la modélisation secondaire ;
- conversion défensive des colonnes numériques ;
- contrôle des doublons sur la clé `Area`–`Item`–`Year`.


In [12]:
missing_after_merge = merged.isna().sum().sort_values(ascending=False).rename("n_missing").to_frame()
display(missing_after_merge)

final_df = merged.dropna(
    subset=[
        "pesticides_tonnes",
        "average_rain_fall_mm_per_year",
        "avg_temp",
    ]
).copy()

num_cols = [
    "Year",
    "hg/ha_yield",
    "pesticides_tonnes",
    "average_rain_fall_mm_per_year",
    "avg_temp",
]

for col in num_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

print("Shape après suppression des lignes inexploitables :", final_df.shape)
print("Doublons exacts :", final_df.duplicated().sum())
print("Doublons sur Area/Item/Year :", final_df.duplicated(subset=["Area", "Item", "Year"]).sum())

display(final_df[num_cols].describe().T)


,n_missing
pesticides_tonnes,32564
average_rain_fall_mm_per_year,31317
avg_temp,24507
Area,0
Unit,0
Year,0
Item,0
hg/ha_yield,0


Shape après suppression des lignes inexploitables : (13136, 8)
Doublons exacts : 0
Doublons sur Area/Item/Year : 0


,count,mean,std,min,25%,50%,75%,max
Year,13136.0,2001.617692,7.035556,1990.00,1995.0000,2001.00,2008.00,2013.00
hg/ha_yield,13136.0,70959.634440,79111.427638,50.00,18000.0000,39536.00,97159.00,501412.00
pesticides_tonnes,13136.0,14832.141503,33646.165696,0.04,264.5825,2170.40,13335.22,367778.00
average_rain_fall_mm_per_year,13130.0,1157.238766,743.622488,51.00,608.0000,1083.00,1651.00,3240.00
avg_temp,13136.0,19.839209,6.657741,1.30,15.6700,20.62,25.78,30.42


## 9. Comparaison avec le fichier de référence déjà fusionné

Le fichier `yield_df.csv` n’est pas utilisé comme source principale du pipeline, mais il constitue un point de comparaison utile.
L’objectif est de vérifier que la reconstruction à partir des sources élémentaires conduit à un résultat cohérent.


In [13]:
yield_ref_clean = yield_ref.drop(columns=["Unnamed: 0"], errors="ignore").copy()

keys_final = set(map(tuple, final_df[["Area", "Item", "Year"]].to_numpy()))
keys_ref = set(map(tuple, yield_ref_clean[["Area", "Item", "Year"]].to_numpy()))

comparison_df = pd.DataFrame([
    {"metric": "n_rows_final_df", "value": final_df.shape[0]},
    {"metric": "n_rows_yield_ref_clean", "value": yield_ref_clean.shape[0]},
    {"metric": "n_common_keys", "value": len(keys_final & keys_ref)},
    {"metric": "n_keys_only_in_final_df", "value": len(keys_final - keys_ref)},
    {"metric": "n_keys_only_in_yield_ref_clean", "value": len(keys_ref - keys_final)},
])

display(comparison_df)


,metric,value
0,n_rows_final_df,13136
1,n_rows_yield_ref_clean,28242
2,n_common_keys,13130
3,n_keys_only_in_final_df,6
4,n_keys_only_in_yield_ref_clean,0


In [14]:
display(
    final_df[[
        "hg/ha_yield",
        "average_rain_fall_mm_per_year",
        "pesticides_tonnes",
        "avg_temp",
    ]].describe().T
)

display(
    yield_ref_clean[[
        "hg/ha_yield",
        "average_rain_fall_mm_per_year",
        "pesticides_tonnes",
        "avg_temp",
    ]].describe().T
)


,count,mean,std,min,25%,50%,75%,max
hg/ha_yield,13136.0,70959.634440,79111.427638,50.00,18000.0000,39536.00,97159.00,501412.00
average_rain_fall_mm_per_year,13130.0,1157.238766,743.622488,51.00,608.0000,1083.00,1651.00,3240.00
pesticides_tonnes,13136.0,14832.141503,33646.165696,0.04,264.5825,2170.40,13335.22,367778.00
avg_temp,13136.0,19.839209,6.657741,1.30,15.6700,20.62,25.78,30.42


,count,mean,std,min,25%,50%,75%,max
hg/ha_yield,28242.0,77053.332094,84956.612897,50.00,19919.2500,38295.00,104676.75,501412.00
average_rain_fall_mm_per_year,28242.0,1149.055980,709.812150,51.00,593.0000,1083.00,1668.00,3240.00
pesticides_tonnes,28242.0,37076.909344,59958.784665,0.04,1702.0000,17529.44,48687.88,367778.00
avg_temp,28242.0,20.542627,6.312051,1.30,16.7025,21.51,26.00,30.65


## 10. Export du dataset annexe consolidé

Le fichier final est exporté dans `data/processed/` afin d’alimenter directement le notebook de modélisation du modèle 2.
Cette étape marque la fin du travail de préparation spécifique aux sources du dataset annexe.


In [15]:
output_path = PROCESSED_DIR / "yield_dataset_consolidated.csv"
final_df.to_csv(output_path, index=False)

print(f"Fichier exporté : {output_path}")
print("Shape finale :", final_df.shape)
display(final_df.head())


Fichier exporté : C:\Users\thoma\Documents\Openclassroom\Projet-12\data\processed\yield_dataset_consolidated.csv
Shape finale : (13136, 8)


,Area,Item,Year,Unit,hg/ha_yield,pesticides_tonnes,average_rain_fall_mm_per_year,avg_temp
253,Albania,Maize,1990,hg/ha,36613,121.0,1485.0,16.37
254,Albania,Maize,1991,hg/ha,29068,121.0,1485.0,15.36
255,Albania,Maize,1992,hg/ha,24876,121.0,1485.0,16.06
256,Albania,Maize,1993,hg/ha,24185,121.0,1485.0,16.05
257,Albania,Maize,1994,hg/ha,25848,201.0,1485.0,16.96


## Conclusion

Le dataset annexe consolidé a été reconstruit à partir des sources élémentaires, harmonisé puis exporté dans le dossier `processed`.

Ce notebook remplit trois fonctions dans la démarche globale :

1. documenter les sources mobilisées pour le dataset annexe ;
2. justifier les choix de nettoyage et de fusion ;
3. produire le fichier utilisé ensuite dans `04_modeling_secondary_recommendation.ipynb`.

L’exploration détaillée du dataset principal et les analyses multivariées relèvent du notebook `02_eda_analysis.ipynb`.
